# Tag2Vec — Training
Reads `tags.csv`, trains skip-gram on tag co-occurrence, saves `tag_embeddings.json`.

In [76]:
import numpy as np
import os
import json
import re
import torch
import torch.nn as nn
from collections import defaultdict

## Config

In [77]:
DATA_DIR    = './data/'
CSV_DIR     = DATA_DIR + 'csv/'
TAGS_PATH   = CSV_DIR + 'tags.csv'
OUTPUT_PATH = DATA_DIR + 'embeddings/tag_embeddings.json'

MIN_TAG_SONGS = 100
MIN_TAG_POP   = 5
TAG_POP_CAP   = 5

MIN_FREQ    = 3
WINDOW      = 4
EMBED_DIM   = 64
NEG_SAMPLES = 5
LR          = 0.001
BATCH_SIZE  = 4096
EPOCHS      = 100
PATIENCE    = 10
MIN_DELTA   = 0.001

VALID_CATEGORIES = {
    "melancholic", "energetic", "aggressive", "chill", "romantic", "dark",
    "happy", "sad", "atmospheric", "dreamy", "peaceful", "angry", "uplifting",
    "nostalgic", "intense", "relaxing", "euphoric", "haunting", "playful",
    "epic", "mysterious", "sensual", "raw", "emotional", "powerful",
    "acoustic", "electronic", "instrumental", "vocal", "heavy", "soft",
    "distorted", "ambient", "lo-fi", "psychedelic", "funky", "groovy",
    "jazzy", "bluesy", "soulful", "punky", "folky", "orchestral",
    "slow", "fast", "uptempo", "downtempo", "driving", "hypnotic", "rhythmic",
    "jazz", "blues", "rock", "pop", "classical", "folk", "metal", "hip hop",
    "reggae", "soul", "funk", "disco", "country", "latin",
    "beach", "summer", "winter", "night", "rainy",
}

BLOCKLIST = {
    # Favourites / personal collections
    "allboutguitar", "aitchsonic", "aitch", "abc", "add", "balida", "amor",
"awsome", "actresses", "ccm", "anime", "animals", "afternoon", "abstract",
"anthems", "accoustic", "again",
    "favorites", "favourite", "favourites", "favorite", "loved", "love at first listen",
    "favorite songs", "personal favourites", "good stuff", "all the best", "fav", "faves",
    "fave", "my favorite", "all time favorites", "all time favourites", "all time fav",
    "all time favourite", "my favourites", "my favorites", "my favorite songs", "my faves",
    "my favs", "my fav", "my best", "my essential songs", "fav songs", "fav tracks",
    "favourite songs", "favourite tracks", "loved tracks", "songs i love",
    "songs i absolutely love", "lovedtrack", "lovedbybeyondwithin", "on repeat",
    "repeatrepeatrepeat", "to download", "to buy", "to get", "to find again",
    "to check out", "to listen", "to classify", "wishlist", "wish list", "collection",
    "recos", "check out", "check it out", "check it", "check this out", "check",
    "investigate", "look into", "find more", "listen", "listen again", "discover",
    "get it later", "download", "buy", "recommended", "essentials", "best of",
    "best of 2006", "best of 2007", "best of 2008", "best of 2009", "best of 2010",
    "best of 2011", "best of 2012", "best of 2013", "best of 2014", "best of 2015",
    "best of 2016", "best songs ever", "best song ever", "best songs of the 00s",
    "best songs of the 60s", "best songs of the 70s", "best songs of the 80s",
    "best songs of the 90s", "best ever", "best music ever", "all-time favorites",
    "all-time favourites", "favourite song", "favorite song", "top artists",
    "favorite artists", "favourite artists", "favorite bands", "fave bands",
    "fave songs ever",

    # Ratings / stars / scores
    "8 of 10 stars", "6 of 10 stars", "10 of 10 stars", "4 stars", "7 of 10 stars",
    "9 of 10 stars", "3", "4", "1", "2", "5", "6", "7", "8", "9", "5 stars",
    "4 of 10 stars", "3 stars", "5 of 10 stars", "2 of 10 stars", "3 star", "4 star",
    "5 star", "3 - sterne", "4-sterne", "3 of 10 stars", "yet another 4", "1 star",
    "4 star", "3 star",

    # Usernames / station names
    "vugube62", "eclectonia", "friendsofthekingofrummelpop", "zaplovedtracks", "slgdm",
    "slgdmbestof", "slgdmblues", "slgdmrelaxing", "lovedbygdchill", "femalevocalistsgdchill",
    "singer-songwritergdchill", "gdchills90s", "amayzes loved", "hopuke42", "dsj-loved-tracks",
    "darkestrose-loved tracks", "pivudo45", "giusychevola e che ama",
    "giusychevola gifts from friends", "solomusika-loved", "lovedbyme1", "lovedbyme",
    "arguman-loved tracks", "dakos hall of fame", "kattis hall of fame", "pxs: loved",
    "akirahoshi unsure", "akirahoshi dark", "leapsandloved", "leapsandbeyondwithin",
    "leapsandleeloo", "leapsandbounds favorite songs", "porieux-loved",
    "theo73 loves this music", "beyondwithins party", "metrohadriani loves this music",
    "majors beloved fm", "mpsvdloved", "simenu22", "djpman-loved-tracks",
    "djmfmylovedmusic", "rustycanuckloved", "lizvelrene2010", "lizvelrene2009",
    "lizvelrene loves", "lizvelrene postpunk", "k1r7m", "k1mo likes", "crybs choice",
    "gertski pick", "annymix", "boyamaca", "davaho53", "christian alexander tietgen",
    "conde maquina favourite track", "meabsolutfav", "miianens playlist", "z3po like this",
    "wolo999", "aproragadozo loves this music", "lara-loved", "transitglambat",
    "whiffer unbound", "whiffer top-notch", "whunderplayed", "igorfree", "hasenradio",
    "jills station", "radioultra", "radio4265", "radio-andree", "audioase", "audioeric-fm",
    "ffmradio", "ffmradionew", "groovesalad", "somafm", "fip", "tantotempotaste",
    "torquemada", "cjl original library", "ion b radio", "ion b chill station",

    # Platform / chart / source tags
    "heard on pandora", "heard on last-fm", "heard on lastfm", "heard on last-fm 09",
    "heard on last-fm 08", "radio paradise", "radioparadise", "pitchfork 500",
    "pitchfork top 500 tracks of the 2000s", "rolling stone 500 greatest songs of all time",
    "rolling stones top 500 songs of all time", "1001 songs you must hear before you die",
    "songs to hear before you die", "rnr hall of fame 500 songs that shaped rock and roll",
    "acclaimed music top 3000", "acclaimed music top 3000 bubbling under", "rs 500", "rs500",
    "top 2000", "top2000", "top1000", "billboard number ones", "uk number one",
    "us number one", "uk top 40", "fm4", "kexp song of the day", "radio disney", "radio",
    "bbc radio1 playlist 2016", "bbc radio1 playlist 2015", "bbc radio1 playlist 2014",
    "bbc6", "john peel", "gilles peterson", "thesixtyone", "itunes uk single of the week",
    "itunes", "spotify", "radio karlsruhe", "1live fiehe", "kat fm", "lautfm bluesclub",
    "wwwlautfmbluesclub", "cross rhythms", "united christian broadcasters", "radiou",
    "wjlb-fm", "wkqi-fm", "cimx-fm", "wber", "whtd-fm", "wrif-fm", "kdwb",

    # Years / decades
    "00s", "80s", "90s", "60s", "70s", "2000s", "10s", "1980s", "1990s", "1970s",
    "1960s", "2010s", "50s", "80's", "90's", "60's", "70's", "80er", "90er", "70er",
    "2008", "2011", "2010", "2009", "2012", "2007", "2013", "2014", "2006", "2005",
    "2004", "2003", "2002", "2001", "2000", "2016", "2015", "1996", "1995", "1999",
    "1998", "1997", "1994", "1993", "1992", "1991", "1990", "1989", "1988", "1987",
    "1986", "1985", "1984", "1983", "1982", "1981", "1980", "1979", "1978", "1977",
    "1976", "1975", "1974", "1973", "1972", "1971", "1970", "1969", "1968", "1967",
    "1966", "1965", "1964", "1963", "1962", "1961", "1960", "1959", "1958", "1957",
    "1956", "1950s", "1920s", "30s", "40s", "50's", "sixties", "seventies", "eighties",
    "nineties", "21st century", "20th century", "before the 70s",

    # Nationality / geography
    "american", "british", "german", "french", "swedish", "canadian", "australian",
    "norwegian", "finnish", "irish", "spanish", "italian", "japanese", "russian",
    "dutch", "danish", "turkish", "polish", "belgian", "icelandic", "brazilian",
    "south african", "korean", "portuguese", "romanian", "ukrainian", "hungarian",
    "greek", "czech", "welsh", "swiss", "scottish", "arabic", "chinese", "indian",
    "persian", "bulgarian", "indonesian", "georgian", "israeli", "mexican",
    "usa", "uk", "us", "england", "france", "germany", "norway", "sweden", "finland",
    "ireland", "spain", "italy", "japan", "russia", "netherlands", "denmark",
    "belgium", "iceland", "brazil", "australia", "new zealand", "scotland", "wales",
    "austria", "switzerland", "poland", "ukraine", "mexico", "india", "turkey",
    "korea", "israel", "california", "texas", "chicago", "new york", "new york city",
    "nyc", "seattle", "boston", "detroit", "portland", "san francisco", "brooklyn",
    "los angeles", "la", "london", "berlin", "paris", "hamburg", "manchester",
    "philadelphia", "philly", "africa", "african", "europe", "argentina", "colombia",
    "chile", "peru", "cuba", "jamaica", "caribbean",

    # Nonsense / gibberish / random strings
    "dd", "aaa", "lol", "wtf", "ok", "yay", "wow", "yeah", "yes", "omg", "hmm",
    "hmmm", "fuck", "fuck yeah", "fucking awesome", "fucking incredible",
    "fucking brilliant", "fucking great", "fucking love", "fuck you", "fuck music",
    "good shit", "shit", "crap", "lame", "awful", "trash", "annoying", "bad", "silly",
    "ridiculous", "overrated", "stupid", "boring", "dope", "rad", "swag", "kick ass",
    "kickass", "badass", "hell yeah", "damn", "hell", "xxx", "vagina", "vaginal",
    "sex", "porn", "bdsm", "erotic", "weed", "marijuana", "ganja", "drugs", "420",
    "drunk", "alcohol", "beer", "whiskey",
    "a", "b", "c", "d", "e", "f", "g", "h", "j", "l", "m", "o", "p", "r", "s", "t",
    "x", "y", "131", "600", "123", "100", "42", "80", "70", "60", "500", "999",
    "0001", "0004",

    # Generic commentary / filler
    "good", "nice", "cool", "awesome", "amazing", "beautiful", "perfect", "brilliant",
    "great", "best", "love", "loved", "like", "loved", "interesting", "unique",
    "special", "original", "fresh", "new", "popular", "mainstream", "timeless",
    "legendary", "influential", "seminal", "underrated", "overrated", "recommended",
    "masterpiece", "perfection", "genius", "superb", "fantastic", "flawless",
    "glorious", "gorgeous", "wonderful", "stunning", "incredible", "magnificent",
    "godlike", "good song", "great song", "awesome song", "beautiful song",
    "amazing song", "best song", "cool song", "pretty good", "very good", "not bad",
    "really good", "excellent", "damned good", "fucking great", "good music",
    "good memories", "good times", "good morning", "good mood",
    "makes me smile", "makes me happy", "makes me cry", "makes me want to dance",
    "makes me wanna dance", "feel good music", "songs i like", "life is easy",
    "everything i need", "my life", "everything",

    # TV shows / movies / games / brands
    "gossip girl", "one tree hill", "greys anatomy", "grey's anatomy", "scrubs",
    "the vampire diaries", "vampire diaries", "smallville", "skins", "teen wolf",
    "grand theft auto", "gta", "gta san andreas", "gta sa", "gta vice city",
    "guitar hero", "rock band", "rock band dlc", "rock band network", "doctor who",
    "daria", "twilight", "the oc", "american idol", "x-factor", "the x factor",
    "itunes", "spotify", "coldplay", "metallica", "muse", "radiohead", "depeche mode",
    "selena gomez", "hilary duff", "vanessa hudgens", "megadeth", "queen",
    "michael jackson",

    # Misc filler
    "test", "misc", "other", "all", "get", "up", "down", "stream", "play", "mix",
    "song", "music", "genre", "tag", "names", "numbers", "single", "singles", "hits",
    "playlist", "covers", "cover", "cover song", "cover version", "cover songs",
    "remix", "tribute", "sample", "samples", "sampled", "sampling", "personal",
}


In [78]:
assert os.path.exists(TAGS_PATH), f"Missing file: {TAGS_PATH}"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

## 1. Load & Filter Tags

In [79]:
def read_csv(path):
    with open(path) as f:
        lines = f.read().strip().splitlines()
    headers = [h.strip('"') for h in lines[0].split(',')]
    rows = []
    for line in lines[1:]:
        parts = [p.strip('"') for p in line.split(';')]
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))
    return rows

def is_valid_tag(tag):
    t = tag.lower().strip()
    if t in BLOCKLIST: return False
    if t in VALID_CATEGORIES: return True
    if (len(tag.split()) == 1
        and tag.islower()
        and tag.isascii()
        and tag.isalpha()
        and 3 <= len(tag) <= 20):
        return True
    return False

tag_freq = defaultdict(list)
for row in read_csv(TAGS_PATH):
    tag_freq[row["tag"]].append(int(row["popularity"]))

valid_tags = {
    tag for tag, pops in tag_freq.items()
    if len(pops) >= MIN_TAG_SONGS
    and np.mean(pops) > MIN_TAG_POP
    and is_valid_tag(tag)
}
print(f"Valid tags: {len(valid_tags)} / {len(tag_freq)} total")
print(f"Sample: {sorted(list(valid_tags))[:20]}")


Valid tags: 985 / 155145 total
Sample: ['Beach', 'Classical', 'Disco', 'Dreamy', 'Driving', 'Energetic', 'Lo-Fi', 'Playful', 'Uplifting', 'accordion', 'acoustic', 'addicting', 'addictive', 'adorable', 'afrobeat', 'aggressive', 'aggro', 'alone', 'alt', 'alternative']


## 2. Build Documents

In [80]:
docs = defaultdict(set)
for row in read_csv(TAGS_PATH):
    if row["tag"] not in valid_tags: continue
    weight = min(int(row["popularity"]), TAG_POP_CAP)
    for _ in range(weight):
        docs[row["song_spotify_id"]].add(row["tag"])

documents = [list(tags) for tags in docs.values() if len(tags) >= 2]
print(f"{len(documents):,} songs | {sum(len(d) for d in documents):,} total tokens")

28,354 songs | 222,455 total tokens


## 3. Vocabulary

In [81]:
freq = defaultdict(int)
for doc in documents:
    for t in doc: freq[t] += 1

idx2token = [t for t, c in sorted(freq.items()) if c >= MIN_FREQ]
token2idx = {t: i for i, t in enumerate(idx2token)}
V = len(idx2token)
print(f"{V} tags in vocab")

985 tags in vocab


## 4. Skip-gram Pairs

In [82]:
pairs = []
for doc in [[token2idx[t] for t in d if t in token2idx] for d in documents]:
    if len(doc) < 2: continue
    for i, c in enumerate(doc):
        for j in range(max(0, i - WINDOW), min(len(doc), i + WINDOW + 1)):
            if j != i: pairs.append((c, doc[j]))

pairs = np.array(pairs, dtype=np.int32)
print(f"{len(pairs):,} training pairs")

1,263,358 training pairs


## 5. Train Tag2Vec

In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

class Tag2Vec(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.W_in  = nn.Embedding(vocab_size, embed_dim)
        self.W_out = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.W_in.weight,  -0.1, 0.1)
        nn.init.constant_(self.W_out.weight, 0)

    def forward(self, center, context, negatives):
        vc = self.W_in(center)
        vo = self.W_out(context)
        vn = self.W_out(negatives)
        pos_loss = torch.log(torch.sigmoid((vc * vo).sum(1))).mean()
        neg_loss = torch.log(torch.sigmoid(-(vc.unsqueeze(1) * vn).sum(2))).mean()
        return -(pos_loss + neg_loss)

model = Tag2Vec(V, EMBED_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
pairs_tensor = torch.tensor(pairs, dtype=torch.long)

best_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    idx = torch.randperm(len(pairs_tensor))
    pairs_shuffled = pairs_tensor[idx]
    total_loss = 0.0

    for s in range(0, len(pairs_shuffled), BATCH_SIZE):
        batch     = pairs_shuffled[s:s+BATCH_SIZE].to(device)
        center    = batch[:, 0]
        context   = batch[:, 1]
        negatives = torch.randint(0, V, (len(batch), NEG_SAMPLES), device=device)

        loss = model(center, context, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / (len(pairs_shuffled) // BATCH_SIZE)
    print(f"Epoch {epoch+1}/{EPOCHS} — loss={avg_loss:.4f}")

    if avg_loss < best_loss - MIN_DELTA:
        best_loss = avg_loss
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  no improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

Win = model.W_in.weight.detach().cpu().numpy()
print(f"Done. Best loss: {best_loss:.4f}")

Using: cpu
Epoch 1/100 — loss=1.0810
Epoch 2/100 — loss=0.9119
Epoch 3/100 — loss=0.8570
Epoch 4/100 — loss=0.8215
Epoch 5/100 — loss=0.8028
Epoch 6/100 — loss=0.7903
Epoch 7/100 — loss=0.7831
Epoch 8/100 — loss=0.7766
Epoch 9/100 — loss=0.7724
Epoch 10/100 — loss=0.7689
Epoch 11/100 — loss=0.7657
Epoch 12/100 — loss=0.7628
Epoch 13/100 — loss=0.7604
Epoch 14/100 — loss=0.7580
Epoch 15/100 — loss=0.7557
Epoch 16/100 — loss=0.7546
Epoch 17/100 — loss=0.7530
Epoch 18/100 — loss=0.7513
Epoch 19/100 — loss=0.7496
Epoch 20/100 — loss=0.7486
Epoch 21/100 — loss=0.7475
Epoch 22/100 — loss=0.7463
Epoch 23/100 — loss=0.7449
Epoch 24/100 — loss=0.7440
  no improvement (1/10)
Epoch 25/100 — loss=0.7424
Epoch 26/100 — loss=0.7421
  no improvement (1/10)
Epoch 27/100 — loss=0.7411
Epoch 28/100 — loss=0.7397
Epoch 29/100 — loss=0.7392
  no improvement (1/10)
Epoch 30/100 — loss=0.7385
Epoch 31/100 — loss=0.7374
Epoch 32/100 — loss=0.7370
  no improvement (1/10)
Epoch 33/100 — loss=0.7363
Epoch 34/10

## 6. Save Embeddings

In [84]:
norms    = np.linalg.norm(Win, axis=1, keepdims=True) + 1e-9
Win_norm = Win / norms

embeddings = {token: Win_norm[i].tolist() for i, token in enumerate(idx2token)}

with open(OUTPUT_PATH, "w") as f:
    json.dump(embeddings, f)

print(f"Saved {len(embeddings)} tag embeddings to {OUTPUT_PATH}")

Saved 985 tag embeddings to ./data/embeddings/tag_embeddings.json
